# Live script clips from a notebook

Drive a running **Gloopy** from this notebook: compose over the control API, and *attach*
the notebook as Gloopy's live **Python generator kernel** — redefine a generator in a cell
and a **Live** clip picks it up on the next loop, no restart.

Start Gloopy first (it listens on `127.0.0.1:50051`), then run the cells top to bottom.


## 1. Connect and attach


In [ ]:
import gloopy
from gloopy import Gloopy

g = Gloopy()                 # the control API (add tracks, clips, transport, ...)
k = gloopy.attach()          # attach THIS notebook as the live Python kernel
k


## 2. Define a generator

A generator takes a context (`clip_len_beats`, `seed`, `key_root`, `tempo_bpm`) and returns
notes. Register it with the `@k.generator` decorator.


In [ ]:
@k.generator
def bassline(ctx):
    root = 36 + (ctx.key_root if ctx.key_root >= 0 else 0)
    steps = [0, 0, 7, 5]
    n = int(ctx.clip_len_beats)
    return [gloopy.note(root + steps[b % len(steps)], b, 0.9) for b in range(n)]


## 3. Make a clip and generate it


In [ ]:
t = g.add_synth_track(name='Bass', wave='SAW')
g.add_clip(t, 0, 4)
g.regenerate_clip(t, 0, lang='python', seed=1)   # runs the notebook generator, materialises notes
g.get_clip_notes(t, 0)


## 4. Go live

Mark the clip **Live** and start it looping. Now **edit the generator below and re-run the
cell** — the clip updates on the next pass while it plays. That's the notebook equivalent of
the Emacs/Sly live-image loop.


In [ ]:
g.set_clip_script_live(t, 0, True)
g.set_loop_to_clip(t, 0)
g.play()


In [ ]:
# Re-run this cell after editing to hear the change on the next loop:
@k.generator
def bassline(ctx):
    root = 36 + (ctx.key_root if ctx.key_root >= 0 else 0)
    import random; rng = random.Random(ctx.seed)
    n = int(ctx.clip_len_beats)
    return [gloopy.note(root + rng.choice([0, 3, 5, 7, 10]), b, 0.9) for b in range(n)]


## 5. Stop / detach


In [ ]:
g.stop()
k.detach()   # Gloopy falls back to the clip's cached notes
